In [1]:
from rdkit.Chem import Descriptors, MolFromSmiles
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, balanced_accuracy_score, matthews_corrcoef, f1_score
from pandas import DataFrame, concat
from pickle import load

In [2]:
#создаем словарь из дескприторов структуры
ConstDescriptors = {"HeavyAtomCount" : Descriptors.HeavyAtomCount,
                       "NHOHCount" : Descriptors.NHOHCount,
                       "NOCount" : Descriptors.NOCount,
                       "NumHAcceptors" : Descriptors.NumHAcceptors,
                       "NumHDonors" : Descriptors.NumHDonors,
                       "NumHeteroatoms" : Descriptors.NumHeteroatoms,
                       "NumRotatableBonds" : Descriptors.NumRotatableBonds,
                       "NumValenceElectrons" : Descriptors.NumValenceElectrons,
                       "NumAromaticRings" : Descriptors.NumAromaticRings,
                       "NumAliphaticHeterocycles" : Descriptors.NumAliphaticHeterocycles,
                       "RingCount" : Descriptors.RingCount}

#создаем словарь из физико-химических дескрипторов                            
PhisChemDescriptors = {"MW" : Descriptors.MolWt,
                          "LogP" : Descriptors.MolLogP,
                          "MR" : Descriptors.MolMR,
                          "TPSA" : Descriptors.TPSA}

#объединяем все дескрипторы в один словарь
descriptors = {}
descriptors.update(ConstDescriptors)
descriptors.update(PhisChemDescriptors)
print(len(descriptors), "количество дескрипторов в словаре")


#функция для генерации дескрипторов из молекул
def mol_dsc_calc(mols):
    
    return DataFrame({k: f(m) for k, f in descriptors.items()} 
             for m in mols)


with open('../data/classifier/modeling_data.pickle', 'rb') as file:
    data = load(file)
#оформляем sklearn трансформер для использования в конвеерном моделировании (sklearn Pipeline)
descriptors_transformer = FunctionTransformer(mol_dsc_calc, validate=False)

15 количество дескрипторов в словаре


In [3]:
molecules = [
    MolFromSmiles(mol) for mol in data['SMILES']
]

In [4]:
X = descriptors_transformer.transform(molecules)
Y = data['Activity']
XY = concat((X, Y), axis=1)

In [5]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.20)

In [6]:
scaler_x = StandardScaler().fit(x_train)
x_train_scal = scaler_x.transform(x_train)
x_test_scal = scaler_x.transform(x_test)

In [ ]:
rf = RandomForestClassifier(class_weight='balanced')
pg = {'n_estimators': [50, 100, 200, 300, 400, 500],
      'max_depth': [None, 1, 3, 5, 7, 10],
      }
cv = RepeatedKFold(n_splits=5, n_repeats=5)
gs = GridSearchCV(rf, pg, verbose=3, cv=cv, refit='balanced_accuracy', scoring=('f1', 'balanced_accuracy'))

gs.fit(x_train_scal, y_train)

Fitting 25 folds for each of 36 candidates, totalling 900 fits
[CV 1/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.688) f1: (test=0.954) total time=   3.3s
[CV 2/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.695) f1: (test=0.957) total time=   3.2s
[CV 3/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.695) f1: (test=0.954) total time=   3.2s
[CV 4/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.690) f1: (test=0.955) total time=   3.2s
[CV 5/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.691) f1: (test=0.955) total time=   3.2s
[CV 6/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.691) f1: (test=0.956) total time=   3.3s
[CV 7/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.692) f1: (test=0.954) total time=   3.2s
[CV 8/25] END max_depth=None, n_estimators=50; balanced_accuracy: (test=0.690) f1: (test=0.955) total time=   3.2s
[CV 9/25] END max

In [8]:
y_pred = gs.predict(x_test_scal)

In [9]:
gs.best_estimator_

RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=200)

In [10]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel().tolist()
precision = tp / (tp + fp)
recall = tp / (tp + fn)

In [11]:
print(f'ACC = {accuracy_score(y_test, y_pred):.3f}')
print(f'BA = {balanced_accuracy_score(y_test, y_pred):.3f}')
print(f'MCC = {matthews_corrcoef(y_test, y_pred):.3f}')
print(f'ROC AUC = {roc_auc_score(y_test, y_pred):.3f}')
print(f'F1 = {f1_score(y_test, y_pred):.3f}')
print(f'Precision = {precision:.3f}')
print(f'Recall = {recall:.3f}')

ACC = 0.841
BA = 0.819
MCC = 0.474
ROC AUC = 0.819
F1 = 0.905
Precision = 0.972
Recall = 0.847
